# Prompt Engineering Advanced

## Meta-Prompting

Ask the LLM to **generate or optimize its own prompt** before answering:

```
Your task is to improve the following prompt for a GPT model, then use the improved prompt to answer.
Original prompt: "Explain neural networks"
```

---

## Automatic Prompt Engineer (APE)

Automatically search for the best instruction using the LLM itself:

1. Generate candidate instructions from input-output demos
2. Score each instruction on a held-out set
3. Select the highest-scoring instruction

$$\text{APE} = \arg\max_\rho \mathbb{E}_{(x,y) \sim D}[\log P(y \mid \rho, x)]$$

---

## Prompt Chaining

Break complex tasks into a **pipeline of prompts** where each output feeds the next:

```
Step 1: Extract key facts from document → {facts}
Step 2: Identify controversies in {facts} → {controversies}  
Step 3: Write a balanced analysis of {controversies}
```

---

## Constitutional AI Prompting

Anthropic's approach: the model critiques and revises its own outputs against a set of principles:

1. **Critique**: "Identify ways the response could be harmful"
2. **Revision**: "Rewrite the response to fix the issues identified"

---

## Advanced Techniques

| Technique | Description |
|-----------|-------------|
| **Skeleton-of-Thought** | Generate answer outline first, fill in parallel |
| **Thread-of-Thought** | Chaotic context → walk through step-by-step |
| **Maieutic Prompting** | Iteratively ask model to explain its reasoning |
| **Contrastive CoT** | Include both correct and incorrect CoT examples |
| **Active Prompting** | Select most uncertain examples for annotation |
| **Emotion Prompting** | Add emotional stimuli ("This is important to my career") |
| **Role Prompting** | Assign expert persona ("You are a senior ML engineer") |

---

## DSPy Programmatic Prompting

DSPy replaces manual prompt strings with **declarative signatures** and **trainable modules**:

```python
class QA(dspy.Signature):
    """Answer questions with short factoid answers."""
    question = dspy.InputField()
    answer   = dspy.OutputField(desc="often between 1 and 5 words")
```

DSPy **optimizers** (teleprompters) automatically find the best prompts/few-shot examples:
- `BootstrapFewShot` auto-generate demonstrations
- `MIPRO` multi-stage instruction and prefix optimization
- `BayesianSignatureOptimizer`

---

## Structured Outputs with Instructor

Instructor wraps OpenAI/Anthropic to return validated **Pydantic models**:

```python
class UserInfo(BaseModel):
    name: str
    age: int
    
user = client.chat.completions.create(..., response_model=UserInfo)
# Returns a validated UserInfo instance, not a string
```

In [1]:
# pip install dspy-ai instructor openai
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ── Prompt Chaining ───────────────────────────────────────────────────────────
def chain(steps: list, initial_input: str) -> str:
    current = initial_input
    for i, step_prompt in enumerate(steps, 1):
        prompt = step_prompt.replace("{input}", current)
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300
        )
        current = r.choices[0].message.content
        print(f"Step {i} output: {current[:100]}...\n")
    return current

document = "AI is transforming healthcare. New models can detect cancer with 95% accuracy."

steps = [
    "Extract 3 key facts from this text:\n{input}",
    "For each fact, identify one potential concern:\n{input}",
    "Write a balanced 2-sentence conclusion addressing these concerns:\n{input}"
]

final = chain(steps, document)

In [2]:
# ── Constitutional AI Self-Critique ───────────────────────────────────────────
def constitutional_revision(initial_response: str, principles: list[str]) -> str:
    critique_prompt = f"""
    Response: {initial_response}
    
    Critique this response against these principles:
    {chr(10).join(f'- {p}' for p in principles)}
    
    Identify specific issues.
    """
    critique = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": critique_prompt}],
        max_tokens=200
    ).choices[0].message.content
    
    revision_prompt = f"""
    Original: {initial_response}
    Critique: {critique}
    
    Rewrite the response to fix all identified issues.
    """
    return client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": revision_prompt}],
        max_tokens=200
    ).choices[0].message.content

initial = "AI will take all jobs and make humans useless."
principles = ["Be accurate and nuanced", "Avoid alarmism", "Acknowledge uncertainty"]
revised = constitutional_revision(initial, principles)
print("Revised:", revised)

In [3]:
# ── Instructor: Structured Outputs ────────────────────────────────────────────
# pip install instructor
import instructor
from pydantic import BaseModel, Field
from typing import List

client_inst = instructor.from_openai(OpenAI(api_key=os.getenv("OPENAI_API_KEY")))

class Entity(BaseModel):
    name: str
    entity_type: str = Field(description="PERSON, ORG, LOCATION, DATE")
    confidence: float = Field(ge=0.0, le=1.0)

class ExtractionResult(BaseModel):
    entities: List[Entity]
    summary: str = Field(description="One sentence summary")

result = client_inst.chat.completions.create(
    model="gpt-4o-mini",
    response_model=ExtractionResult,
    messages=[{
        "role": "user",
        "content": "Extract entities from: 'Elon Musk founded SpaceX in 2002 in Hawthorne, California.'"
    }]
)

for entity in result.entities:
    print(f"{entity.name} ({entity.entity_type}) confidence: {entity.confidence}")
print("Summary:", result.summary)

API call failed on attempt 1: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy*****************xxxx. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


Max retries exceeded. Total attempts: 1, Last error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy*****************xxxx. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


In [4]:
# ── DSPy Basic Example ────────────────────────────────────────────────────────
# pip install dspy-ai
import dspy

lm = dspy.LM("openai/gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))
dspy.configure(lm=lm)

class SentimentSignature(dspy.Signature):
    """Classify sentiment of a product review."""
    review: str = dspy.InputField()
    sentiment: str = dspy.OutputField(desc="positive, negative, or neutral")
    confidence: float = dspy.OutputField(desc="confidence score between 0 and 1")

classify = dspy.Predict(SentimentSignature)
result = classify(review="The battery life is incredible, best phone I've owned!")
print(f"Sentiment: {result.sentiment}, Confidence: {result.confidence}")

2026/06/19 16:26:29 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


## Additional Learning Resources

### Papers
- [Automatic Prompt Engineer (Zhou et al., 2022)](https://arxiv.org/abs/2211.01910)
- [DSPy (Khattab et al., 2023)](https://arxiv.org/abs/2310.03714)
- [Constitutional AI (Bai et al., 2022)](https://arxiv.org/abs/2212.08073)
- [Skeleton-of-Thought (Ning et al., 2023)](https://arxiv.org/abs/2307.15337)
- [EmotionPrompt (Li et al., 2023)](https://arxiv.org/abs/2307.11760)

### Frameworks
- [DSPy Docs](https://dspy.ai/)
- [Instructor Library](https://python.useinstructor.com/)
- [Guardrails AI](https://www.guardrailsai.com/docs)

### Guides
- [Prompt Engineering Guide Advanced](https://www.promptingguide.ai/techniques/prompt_chaining)
- [Anthropic Cookbook](https://github.com/anthropics/anthropic-cookbook)